[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/finetuning-avanzado/01-dpo-rlhf.ipynb)

# DPO y RLHF: Alineación de Modelos por Preferencias

> Aprende las técnicas de alineación de LLMs con preferencias humanas: RLHF clásico
> y el más moderno DPO. Genera datasets de preferencias automáticamente con Claude.

## Objetivos
- Entender la diferencia entre RLHF y DPO
- Generar un dataset de preferencias con Claude Haiku
- Configurar DPOTrainer de TRL para fine-tuning
- Evaluar la diferencia entre modelo base y alineado

**Nota:** El entrenamiento real requiere GPU. Este notebook muestra la arquitectura completa
con un modelo mínimo ejecutable en CPU para fines educativos.

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic trl transformers datasets torch --quiet

## 2. RLHF vs DPO: conceptos clave

In [ ]:
import anthropic
import json
import pandas as pd

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

comparativa = pd.DataFrame([
    {"Técnica": "RLHF", "Pasos": "3 (SFT → Reward Model → PPO)", "Complejidad": "Alta", "Estabilidad": "Media", "Datos necesarios": "Comparaciones de preferencias"},
    {"Técnica": "DPO", "Pasos": "1 (entrenamiento directo)", "Complejidad": "Baja", "Estabilidad": "Alta", "Datos necesarios": "Pares (elegido, rechazado)"},
    {"Técnica": "ORPO", "Pasos": "1 (SFT + preferencias)", "Complejidad": "Baja", "Estabilidad": "Alta", "Datos necesarios": "Pares (elegido, rechazado)"},
])
print("=== COMPARATIVA RLHF vs DPO vs ORPO ===")
print(comparativa.to_string(index=False))

print("""
Flujo RLHF:
  1. SFT:          Modelo base → fine-tune supervisado con ejemplos de calidad
  2. Reward Model: Entrenar un modelo que puntúa qué respuesta es mejor
  3. PPO:          Usar el reward model para reforzar el modelo SFT con RL

Flujo DPO (más simple):
  1. Dataset de pares: (prompt, respuesta_elegida, respuesta_rechazada)
  2. Optimización directa de la política usando una función de pérdida DPO
     → No necesita reward model separado
""")

## 3. Generar dataset de preferencias con Claude

In [ ]:
PROMPTS_BASE = [
    "Explica qué es el machine learning en 2 frases.",
    "¿Cuál es la diferencia entre clasificación y regresión?",
    "¿Qué es el overfitting y cómo se evita?",
    "Describe las redes neuronales convolucionales.",
    "¿Para qué sirven los embeddings?"
]

def generar_par_preferencias(prompt: str) -> dict:
    """Claude genera respuesta elegida (buena) y rechazada (mala) para un prompt."""
    instruccion = f"""Para el siguiente prompt de IA, genera:
1. Una respuesta ELEGIDA: precisa, clara, educativa, bien estructurada
2. Una respuesta RECHAZADA: vaga, imprecisa o de baja calidad

Prompt: "{prompt}"

Responde SOLO con JSON:
{{"prompt": "{prompt}", "chosen": "<respuesta buena>", "rejected": "<respuesta mala>"}}"""

    resp = client.messages.create(
        model=MODELO, max_tokens=512,
        messages=[{"role": "user", "content": instruccion}]
    ).content[0].text.strip()

    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json").strip()
    return json.loads(resp)

print("Generando dataset de preferencias con Claude...")
dataset_preferencias = []
for prompt in PROMPTS_BASE:
    par = generar_par_preferencias(prompt)
    dataset_preferencias.append(par)
    print(f"  ✓ {prompt[:50]}")

print(f"\nDataset generado: {len(dataset_preferencias)} pares de preferencias")
print("\nEjemplo:")
print(f"  Prompt: {dataset_preferencias[0]['prompt']}")
print(f"  Chosen: {dataset_preferencias[0]['chosen'][:100]}...")
print(f"  Rejected: {dataset_preferencias[0]['rejected'][:100]}...")

## 4. Configurar DPOTrainer de TRL

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DPOConfig, DPOTrainer

# Usar GPT-2 como modelo base (pequeño, funciona en CPU)
MODELO_BASE = "gpt2"
print(f"Cargando modelo base: {MODELO_BASE}")

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 no tiene pad token
modelo = AutoModelForCausalLM.from_pretrained(MODELO_BASE)
modelo_referencia = AutoModelForCausalLM.from_pretrained(MODELO_BASE)  # Modelo de referencia (congelado)

# Preparar dataset en formato HuggingFace
dataset_hf = Dataset.from_list(dataset_preferencias)

# Configuración de entrenamiento DPO
config_dpo = DPOConfig(
    output_dir="./dpo_output",
    num_train_epochs=1,          # 1 epoch para demo
    per_device_train_batch_size=1,
    learning_rate=1e-5,
    beta=0.1,                    # Temperatura DPO: controla divergencia del modelo de referencia
    max_length=256,
    max_prompt_length=128,
    logging_steps=1,
    save_strategy="no"           # No guardar checkpoints en demo
)

trainer_dpo = DPOTrainer(
    model=modelo,
    ref_model=modelo_referencia,
    args=config_dpo,
    train_dataset=dataset_hf,
    tokenizer=tokenizer,
)

print("DPOTrainer configurado correctamente.")
print(f"Dataset de entrenamiento: {len(dataset_hf)} ejemplos")
print(f"Beta (temperatura DPO): {config_dpo.beta}")
print("\nNota: El entrenamiento completo requiere GPU y muchas más épocas.")
print("      Para producción, usar modelos como Llama-3 o Mistral con LoRA.")

## 5. Evaluación cualitativa antes del entrenamiento

In [ ]:
import torch

def generar_respuesta(modelo_eval, tokenizer_eval, prompt: str, max_new_tokens: int = 60) -> str:
    """Genera una respuesta con el modelo especificado."""
    inputs = tokenizer_eval(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = modelo_eval.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer_eval.eos_token_id
        )
    return tokenizer_eval.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=== RESPUESTA DEL MODELO BASE (GPT-2, sin DPO) ===")
prompt_test = "Explain what machine learning is:"
resp_base = generar_respuesta(modelo, tokenizer, prompt_test)
print(f"Prompt: '{prompt_test}'")
print(f"Respuesta: {resp_base}")

print("\n=== NOTA SOBRE EL PROCESO COMPLETO ===")
print("Para un DPO completo en producción:")
print("  1. Usar un modelo base de calidad (Llama-3.2, Mistral 7B, Qwen)")
print("  2. Aplicar LoRA/QLoRA para reducir memoria GPU necesaria")
print("  3. Dataset de preferencias: mínimo 1000-10000 pares")
print("  4. Entrenar 1-3 épocas con learning_rate = 1e-6 a 5e-5")
print("  5. Evaluar con benchmarks: MT-Bench, AlpacaEval, Chatbot Arena")